# V6.5-22 — Teacher v5 vs v6 (cùng cấu hình) → hệ 2 tầng → student n cho Pi — chạy một mạch trên GPU thuê

**Mục tiêu:** biết dataset **v6** (v5 + review toàn bộ foreign_object, `data/label_audit/README.md` mục v6) có làm hệ
thống tốt hơn không, đo trên **val_clean_v6** (ảnh vuông như Pi, paired bootstrap). Baseline v5 hiện tại:
hệ 2 tầng (Pi n v5 gắn cờ @0.05, server teacher v5, FO lấy ở 800) = **0.8285** (CI95 0.798–0.860).

**So sánh công bằng:** teacher v5 được fine-tune **2 lần cùng cấu hình** (lr thấp, không warmup, cùng seed/epoch):
một lần trên v5 (đối chứng) và một lần trên v6. Khác nhau duy nhất là dataset.

**Chạy (máy Ubuntu có GPU, ví dụ 3090 24 GB):** `bash scripts/run_v6p5_22.sh` — script cài môi trường, rồi chạy notebook
này bằng nbconvert trong nền; xem tiến độ: `tail -f artifacts/v6p5_v6_teachers/progress.log`. Bị ngắt → chạy lại script:
stage nào xong thì bỏ qua, đang dở thì resume.

| Cell | Việc | 3090 (ước tính) |
|---|---|---|
| 2–3 | tải v5 + weights v5 (Camber), **build v6 từ v5** (`scripts/build_v6.py`), kiểm tra hash nhãn | ~15 phút |
| 5 | teacher: fine-tune trên v5 và trên v6 (`TEACHER_EPOCHS` mỗi lượt) | ~3–4 giờ |
| 6 | đo teacher + hệ 2 tầng, bootstrap v6 − v5 → `eval_teachers.csv` | ~10 phút |
| 7 | student n (từ student v5) distill từ teacher tốt nhất trên v6 → NCNN → đo | ~1.5 giờ |
| 8–9 | upload kết quả + dataset v6 lên Camber, (tuỳ chọn) tắt máy | ~20 phút |

`DRY_RUN = True` (hoặc `DRY_RUN=1`): 1 epoch trên 3% ảnh, đo trên 200 ảnh — để thử cả notebook trong ~10 phút.

In [ ]:
# 1. CẤU HÌNH — mặc định chạy được luôn
import os, sys, platform, shutil, time, json, logging, subprocess
from pathlib import Path

RUN_NAME = "v6p5_v6_teachers"          # artifacts/<RUN_NAME>, trên Camber: <RUN_NAME>_<ngày>
DATASET_NAME = "v6p5_v5"               # tải v5 rồi build v6 tại chỗ (v6 chưa có trên Camber)
STASH_DATASET = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_v5"
STASH_INIT = "stash://bunpmc/projects/yolov11sdi/checkpoints/v6p5_v5_20261013"   # teacher/best.pt, student_distill/best.pt
STASH_RESULTS = "stash://bunpmc/projects/yolov11sdi/checkpoints"
STASH_V6 = "stash://bunpmc/projects/yolov11sdi/datasets/v6p5_v6"
V6_LABELS_SHA256 = "96bd8f659986defb7946480b95843fba648aef9d1487483361daa8aeefeb539b"        # hash nhãn v6 đã duyệt (build trên 4060, 2026-10-15)
ARMS = ["v5", "v6"]                    # v5 = đối chứng (cùng cấu hình), v6 = dataset mới
TEACHER_EPOCHS, TEACHER_LR = 20, 1e-4  # fine-tune teacher v5 (đã hội tụ): lr thấp, không warmup (lr 1e-3 phá model lúc đầu)
TRAIN_STUDENT, STUDENT_EPOCHS = True, 30
TAU = 0.05                             # Pi gắn cờ ảnh khi có box lỗi conf >= TAU
UPLOAD_RESULTS = True                  # cần Camber CLI + CAMBER_API_KEY trong .env (bỏ qua nếu thiếu)
UPLOAD_DATASET_V6 = True               # đóng gói v6 (~6 GB) lên Camber sau khi train xong → lần sau khỏi build
AUTO_SHUTDOWN = False                  # True: `sudo shutdown -h +5` ở cuối (chỉ bật nếu tắt máy = ngừng tính tiền)
DRY_RUN = os.environ.get("DRY_RUN", "0") == "1"

# ---- tự động (không cần sửa) ----
import torch, ultralytics
from ultralytics import YOLO
from ultralytics.utils import LOGGER
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
for line in (REPO / ".env").read_text().splitlines() if (REPO / ".env").is_file() else []:   # CAMBER_API_KEY... (không in ra)
    if "=" in line and not line.lstrip().startswith("#"):
        k, v = line.split("=", 1); os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
DATA_DIR = Path(os.environ.get("V6P5_DATA_DIR", REPO / "datasets"))
OUT = REPO / "artifacts" / (RUN_NAME + ("_dryrun" if DRY_RUN else ""))
RUNS, INIT_DIR = OUT / "runs", OUT / "init"
NAMES = ["insulator", "insulator_broken", "insulator_flashover", "bird_nest", "foreign_object", "broken_strand", "person"]
DEFECTS = NAMES[1:6]
WINDOWS = platform.system() == "Windows"
CAMBER = os.environ.get("CAMBER_BIN") or shutil.which("camber") or str(Path.home() / ".camber/bin" / ("camber.exe" if WINDOWS else "camber"))
HAS_CAMBER = (Path(CAMBER).exists() or bool(shutil.which(CAMBER))) and bool(os.environ.get("CAMBER_API_KEY"))
if ultralytics.__version__ != "8.4.172":
    raise RuntimeError(f"Cần ultralytics==8.4.172 (có distill_model); đang có {ultralytics.__version__}: pip install ultralytics==8.4.172")
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU CUDA (cài torch bản CUDA)")
DEVICE, VRAM = 0, torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH_TEACHER = 16 if VRAM >= 20 else 8 if VRAM >= 11 else 4      # YOLO26m @640, AMP (nbs=64 → gradient tích luỹ như nhau)
BATCH_STUDENT = 32 if VRAM >= 20 else 16 if VRAM >= 11 else 8     # n distill (teacher m chạy kèm mỗi batch)
WORKERS = min(4 if WINDOWS else 8, os.cpu_count() or 2)
OUT.mkdir(parents=True, exist_ok=True); DATA_DIR.mkdir(parents=True, exist_ok=True)
FREE_GB = shutil.disk_usage(DATA_DIR).free / 2**30
CACHE = "disk" if FREE_GB >= 70 and not DRY_RUN else False      # .npy ~55 GB, dùng chung v5/v6 (hard link)
EVAL_N = None
if DRY_RUN:
    TEACHER_EPOCHS = STUDENT_EPOCHS = 1; EVAL_N = 200; UPLOAD_DATASET_V6 = False; AUTO_SHUTDOWN = False
COMMON = dict(device=DEVICE, workers=WORKERS, seed=20261015, deterministic=True, optimizer="AdamW", cos_lr=True, lrf=0.1,
              weight_decay=1e-3, cache=CACHE, amp=True, pretrained=True, plots=False, save_period=5,
              imgsz=320 if DRY_RUN else 640, fraction=0.03 if DRY_RUN else 1.0)

class _Tee:   # mọi print → cả notebook lẫn progress.log (xem khi chạy nền: tail -f progress.log)
    def __init__(self, f, o): self.f, self.o = f, o
    def write(self, x): self.o.write(x); self.f.write(x); self.f.flush(); return len(x)
    def flush(self): self.o.flush(); self.f.flush()
    def __getattr__(self, k): return getattr(self.o, k)
if not isinstance(sys.stdout, _Tee):
    sys.stdout = _Tee(open(OUT / "progress.log", "a", encoding="utf-8"), sys.stdout)
print(f"\n===== {time.strftime('%Y-%m-%d %H:%M')} | {platform.system()} | {torch.cuda.get_device_name(0)} ({VRAM:.0f} GB) | "
      f"ultralytics {ultralytics.__version__} | DRY_RUN {DRY_RUN}")
print(f"teacher batch {BATCH_TEACHER} × {TEACHER_EPOCHS} ep × {ARMS} | student n batch {BATCH_STUDENT} × {STUDENT_EPOCHS} ep "
      f"({'có' if TRAIN_STUDENT else 'không'}) | workers {WORKERS} | cache {CACHE} (ổ trống {FREE_GB:.0f} GB) | camber {HAS_CAMBER}")

In [ ]:
# 2. DATASET: tải v5 từ Camber (bỏ qua nếu đã có) → build v6 từ v5 → kiểm tra hash nhãn
import hashlib, subprocess, tarfile, random, yaml

def camber_cp(src, dst):
    if not (Path(CAMBER).exists() or shutil.which(CAMBER)):
        raise RuntimeError(f"Không thấy Camber CLI ({CAMBER}). Cài CLI hoặc đặt CAMBER_BIN trong .env; "
                           f"hoặc tự tải {src} về {dst}")
    subprocess.run([CAMBER, "stash", "cp", src, str(dst)], check=True, stdout=subprocess.DEVNULL)

DS = DATA_DIR / DATASET_NAME
if not (DS / "manifest.json").is_file():
    tar = DATA_DIR / f"{DATASET_NAME}.tar"
    for f in (f"{DATASET_NAME}.tar.sha256", f"{DATASET_NAME}.package.json", f"{DATASET_NAME}.tar"):
        if not (DATA_DIR / f).is_file():
            print("tải", f, "..."); camber_cp(f"{STASH_DATASET}/{f}", DATA_DIR / f)
    want = (DATA_DIR / f"{DATASET_NAME}.tar.sha256").read_text().split()[0]
    h = hashlib.sha256()
    with tar.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""): h.update(chunk)
    if h.hexdigest() != want:
        tar.unlink(); raise RuntimeError("sha256 không khớp (tải lỗi) → đã xoá file tar, chạy lại cell này")
    print("giải nén ..."); tarfile.open(tar).extractall(DATA_DIR)
    tar.unlink()  # giữ ổ trống cho cache
for y, v in [("data.yaml", "images/val"), ("data_val_clean.yaml", "images/val_clean")]:   # path trong tar là của máy build
    cfg = yaml.safe_load((DS / y).read_text()); cfg.update(path=str(DS), train="images/train", val=v)
    (DS / y).write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
assert [yaml.safe_load((DS / "data.yaml").read_text())["names"][i] for i in range(7)] == NAMES
pkg = json.loads((DATA_DIR / f"{DATASET_NAME}.package.json").read_text())
print("dataset OK:", DS, "| files:", pkg["files"], "| labels_sha256:", pkg["labels_sha256"][:16])


DS6 = DATA_DIR / "v6p5_v6"
if not (DS6 / "manifest.json").is_file():   # build_v6.py: ~5 phút, ảnh hard-link (không tốn thêm ổ)
    if DS6.exists(): shutil.rmtree(DS6)
    print("build v6 từ v5 ..."); t = time.time()
    subprocess.run([sys.executable, str(REPO / "scripts/build_v6.py"), "--v5", str(DS), "--out", str(DS6)], check=True,
                   stdout=subprocess.DEVNULL)
    print(f"build xong {(time.time() - t) / 60:.1f} phút")
m6 = json.loads((DS6 / "manifest.json").read_text())
if m6["labels_sha256"] != V6_LABELS_SHA256:
    raise RuntimeError(f"nhãn v6 khác bản đã duyệt ({m6['labels_sha256'][:12]} ≠ {V6_LABELS_SHA256[:12]}): "
                       "kiểm tra git pull (data/label_audit/fo_review_v6) rồi xoá datasets/v6p5_v6 và chạy lại")
DATA = {"v5": DS / "data.yaml", "v6": DS6 / "data.yaml"}
for y in ["data.yaml", "data_val_clean.yaml", "data_val_clean_v6.yaml"]:   # path trong yaml = máy build
    cfg = yaml.safe_load((DS6 / y).read_text()); cfg["path"] = str(DS6); (DS6 / y).write_text(yaml.safe_dump(cfg, sort_keys=False))

def link_cache():   # ảnh v6 = ảnh v5 → dùng chung cache .npy (hard link), khỏi tạo lại ~55 GB
    n = 0
    for f in (DS / "images/train").glob("*.npy"):
        d = DS6 / "images/train" / f.name
        if not d.exists() and d.with_suffix(".jpg").exists():
            os.link(f, d); n += 1
    if n: print("cache .npy dùng chung v5 → v6:", n)
st = m6["stats"]["train"]
print(f"v6 OK: train {st['images']} ảnh, FO {st['box:foreign_object']}, person {st['box:person']} | "
      f"val_clean_v6 FO {m6['stats']['val_clean_v6']['box:foreign_object']} | hash {V6_LABELS_SHA256[:12]}")

In [ ]:
# 3. WEIGHTS v5 từ Camber: teacher (YOLO26m) + student n (Pi)
INIT_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_V5, STUDENT_V5 = INIT_DIR / "teacher_v5.pt", INIT_DIR / "student_n_v5.pt"
for dst, src_ in [(TEACHER_V5, "teacher/best.pt"), (STUDENT_V5, "student_distill/best.pt")]:
    local = REPO / "artifacts/v6p5_v5_students/init" / dst.name     # máy đã chạy notebook 20 thì có sẵn
    if not dst.is_file():
        if local.is_file(): shutil.copy2(local, dst)
        else: print("tải", src_); camber_cp(f"{STASH_INIT}/{src_}", dst)
    assert list(YOLO(str(dst)).names.values()) == NAMES, dst
print("teacher:", TEACHER_V5, "| student:", STUDENT_V5)

In [ ]:
# 4. HÀM TRAIN (giống notebook 20): resume / bỏ qua nếu xong / watchdog chống treo / tự thử lại, 1 dòng mỗi epoch
import gc, threading, _thread

def _epoch_line(trainer):
    m, loss = trainer.metrics, trainer.label_loss_items(trainer.tloss)
    final = trainer.stop and getattr(trainer, "_stop_seen", False)   # ultralytics val lại best.pt sau epoch cuối
    trainer._stop_seen = trainer.stop
    tag = "best.pt (val cuối)" if final else f"ep {trainer.epoch + 1:>3}/{trainer.epochs}"
    print(f"{trainer.args.name} {tag}  box {float(loss['train/box_loss']):.3f} cls {float(loss['train/cls_loss']):.3f}  "
          f"mAP50 {m.get('metrics/mAP50(B)', 0):.4f}  mAP50-95 {m.get('metrics/mAP50-95(B)', 0):.4f}  "
          f"lr {trainer.optimizer.param_groups[0]['lr']:.2e}  {time.strftime('%H:%M')}", flush=True)

HANG_MINUTES, MAX_RETRIES = 15, 3

class Watchdog:
    """Ngắt train (như bấm Interrupt) nếu không có batch nào chạy xong trong HANG_MINUTES để cell tự resume."""
    def __init__(self, minutes):
        self.limit, self.beat, self.armed, self.fired = minutes * 60, time.time(), False, False
        self._stop = threading.Event()
    def heartbeat(self, *_):
        self.beat, self.armed = time.time(), True
    def _run(self):
        while not self._stop.wait(30):
            if self.armed and time.time() - self.beat > self.limit:
                self.fired = True
                print(f"[watchdog] {self.limit // 60} phút không có batch nào chạy xong → ngắt để resume", flush=True)
                _thread.interrupt_main()   # chạy được cả Windows (không dùng SIGINT)
                return
    def __enter__(self):
        threading.Thread(target=self._run, daemon=True).start(); return self
    def __exit__(self, *exc):
        self._stop.set()

def _latest_checkpoint(run):
    """last.pt, nếu hỏng (bị kill lúc đang ghi) thì lùi về epochN.pt mới nhất."""
    for p in [run / "weights/last.pt", *sorted(run.glob("weights/epoch*.pt"), key=lambda p: -int(p.stem[5:]))]:
        if p.is_file():
            try:
                return p, torch.load(p, map_location="cpu", weights_only=False)
            except Exception as e:
                print(f"[{run.name}] checkpoint hỏng, bỏ qua {p.name}: {e}")
    return None, None

def train_stage(name, model, **kw):
    run, best = RUNS / name, RUNS / name / "weights/best.pt"
    for attempt in range(1, MAX_RETRIES + 2):
        ckpt, ck = _latest_checkpoint(run)
        targs = (ck or {}).get("train_args", {}) or {}
        if ck is not None and (ck.get("epoch", -1) == -1 or ck["epoch"] + 1 >= targs.get("epochs", 10**9)) \
                and targs.get("epochs", kw.get("epochs")) == kw.get("epochs"):
            print(f"[{name}] đã xong, bỏ qua: {best}"); return best
        if ck is not None and targs.get("epochs") != kw.get("epochs"):
            print(f"[{name}] run cũ {targs.get('epochs')} epoch ≠ {kw.get('epochs')} → bắt đầu lại"); ck = None
        if ck is not None:
            print(f"[{name}] resume từ {ckpt.name} (đã xong {ck['epoch'] + 1} epoch)")
            yolo, args = YOLO(str(ckpt)), dict(resume=True)
        else:
            if run.is_dir(): shutil.rmtree(run)
            print(f"[{name}] bắt đầu: {model}")
            yolo, args = YOLO(str(model)), dict(project=str(RUNS), name=name, exist_ok=False, **{**COMMON, **kw})
        wd = Watchdog(HANG_MINUTES)
        for ev in ("on_train_batch_end", "on_val_batch_end", "on_fit_epoch_end"):
            yolo.add_callback(ev, wd.heartbeat)
        yolo.add_callback("on_fit_epoch_end", _epoch_line)
        level = LOGGER.level; LOGGER.setLevel(logging.WARNING)   # tắt progress bar → output gọn
        try:
            with wd:
                yolo.train(**args)
            print(f"[{name}] xong: {best}"); return best
        except KeyboardInterrupt:
            if not wd.fired:
                print(f"[{name}] dừng theo yêu cầu. Chạy lại cell này để resume."); raise
            print(f"[{name}] treo → resume (lần {attempt}/{MAX_RETRIES})")
        except Exception as e:
            print(f"[{name}] lỗi {type(e).__name__}: {e} → resume (lần {attempt}/{MAX_RETRIES})")
        finally:
            LOGGER.setLevel(level); del yolo; gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
        time.sleep(30)
    raise RuntimeError(f"[{name}] thất bại sau {MAX_RETRIES} lần thử lại — xem log ở trên")

In [ ]:
# 5. TEACHER: fine-tune teacher v5 trên từng dataset trong ARMS, CÙNG cấu hình (chỉ khác data)
TEACHER = {"v5_orig": TEACHER_V5}
for arm in ARMS:
    if arm == "v6": link_cache()
    TEACHER[f"ft_{arm}"] = train_stage(f"teacher_{arm}", TEACHER_V5, data=str(DATA[arm]), epochs=TEACHER_EPOCHS,
                                       batch=BATCH_TEACHER, lr0=TEACHER_LR, warmup_epochs=0, close_mosaic=5,
                                       patience=TEACHER_EPOCHS, val=False)   # val=False: chỉ đo ở epoch cuối (nhanh hơn)
print({k: str(v) for k, v in TEACHER.items()})

In [ ]:
# 6. ĐO: teacher-only và hệ 2 tầng (Pi n v5 gắn cờ @TAU; server = teacher, FO lấy ở 800) trên val_clean_v6 (+ val_clean cũ),
#    ảnh vuông như Pi; paired bootstrap trên ảnh cho ft_v6 − ft_v5 (khác nhau chỉ ở dataset) và ft_v6 − v5_orig
import numpy as np, pandas as pd
sys.path.insert(0, str(REPO / "data/label_audit/preflight"))
from system_two_tier import run as predict, flagged
from system_two_tier_bootstrap import prep, wap
B, SEED = 2000, 20261015
IMGS = sorted((DS6 / "images/val_clean_v6").glob("*.jpg"))[:EVAL_N]; IDX = {p.name: i for i, p in enumerate(IMGS)}

def gt(split):
    G = {c: {} for c in range(1, 6)}
    for im in IMGS:
        x = np.array([l.split() for l in (DS6 / "labels" / split / (im.stem + ".txt")).read_text().splitlines() if l.strip()], float).reshape(-1, 5)
        for c in range(1, 6):
            b = x[x[:, 0] == c][:, 1:]; G[c][im.name] = np.c_[b[:, :2] - b[:, 2:] / 2, b[:, :2] + b[:, 2:] / 2]
    return G
GTS = {s: gt(s) for s in ["val_clean_v6", "val_clean"]}
tag = lambda name, sz: f"nb22{'_dry' if DRY_RUN else ''}_{name}_{sz}"
def preds(name, w):   # cache theo đường dẫn weights trong data/label_audit/preflight/sys_<tag>.json
    return {sz: predict(w, IMGS, tag(name, sz), imgsz=sz, device=DEVICE) for sz in (640, 800)}
PI = {"n_v5": predict(STUDENT_V5, IMGS, tag("pi_n_v5", 640), imgsz=640, device=DEVICE)}

def system_scores(teachers, pi_name="n_v5"):
    """-> bảng điểm + mẫu bootstrap MEAN_5 (hệ 2 tầng, server FO@800) cho từng teacher, trên từng split"""
    P = {k: preds(k, w) for k, w in teachers.items()}; flag = flagged(PI[pi_name], TAU); rows, samples = [], {}
    rng = {s: np.random.default_rng(SEED) for s in GTS}
    W = {s: [np.bincount(rng[s].integers(0, len(IMGS), len(IMGS)), minlength=len(IMGS)).astype(float) for _ in range(B)] for s in GTS}
    for s, G in GTS.items():
        for k, p in P.items():
            for setting, keep in [("teacher only", set(IDX)), (f"2 tầng @{TAU}", flag)]:
                pre = [prep(p[800 if c == 4 else 640], G[c], c, keep, IDX) for c in range(1, 6)]
                ap = [wap(*x, np.ones(len(IMGS))) for x in pre]
                rows.append({"split": s, "teacher": k, "setting": setting, "MEAN_5": np.mean(ap), **dict(zip(DEFECTS, ap))})
                if setting != "teacher only":
                    samples[(s, k)] = np.array([np.mean([wap(*x, w) for x in pre]) for w in W[s]])
    return pd.DataFrame(rows), samples

res, S = system_scores(TEACHER)
pd.set_option("display.width", 200)
print(res.pivot_table(index=["split", "setting"], columns="teacher", values="MEAN_5").round(4).to_string())
cmp = []
for s in GTS:
    for a, b in [("ft_v6", "ft_v5"), ("ft_v6", "v5_orig"), ("ft_v5", "v5_orig")]:
        if (s, a) in S and (s, b) in S:
            d = S[(s, a)] - S[(s, b)]; lo, hi = np.percentile(d, [2.5, 97.5])
            cmp.append({"split": s, "compare": f"{a} − {b}", "diff": d.mean(), "ci95_lo": lo, "ci95_hi": hi, "P(>0)": (d > 0).mean()})
cmp = pd.DataFrame(cmp); print("\n" + cmp.round(4).to_string(index=False))
res.to_csv(OUT / "eval_teachers.csv", index=False); cmp.to_csv(OUT / "eval_teachers_compare.csv", index=False)
r = cmp[(cmp.split == "val_clean_v6") & (cmp.compare == "ft_v6 − ft_v5")]
if len(r):
    r = r.iloc[0]; verdict = "v6 TỐT HƠN" if r.ci95_lo > 0 else "v6 KÉM HƠN" if r.ci95_hi < 0 else "CHƯA PHÂN BIỆT ĐƯỢC"
    print(f"\n>>> dataset v6 vs v5 (hệ 2 tầng, val_clean_v6): {r['diff']:+.4f} CI95 [{r.ci95_lo:+.4f}, {r.ci95_hi:+.4f}] → {verdict}")
two = res[(res.split == "val_clean_v6") & (res.setting != "teacher only")].set_index("teacher").MEAN_5
BEST_TEACHER = two.idxmax(); print(f">>> teacher tốt nhất cho hệ 2 tầng: {BEST_TEACHER} ({two.max():.4f}; mục tiêu > 0.82)")

In [ ]:
# 7. STUDENT n cho Pi: từ student v5, distill từ teacher tốt nhất, train trên v6 → NCNN → đo Pi-only + hệ 2 tầng
if TRAIN_STUDENT:
    link_cache()
    STUDENT_NEW = train_stage("student_n_v6", STUDENT_V5, data=str(DATA["v6"]), epochs=STUDENT_EPOCHS, batch=BATCH_STUDENT,
                              lr0=1e-4, warmup_epochs=0, close_mosaic=5, patience=STUDENT_EPOCHS, val=False,
                              distill_model=str(TEACHER[BEST_TEACHER]))
    NCNN = STUDENT_NEW.with_name("best_ncnn_model")
    if not NCNN.is_dir():
        YOLO(str(STUDENT_NEW)).export(format="ncnn", imgsz=640, half=False)   # .pt ảnh vuông = NCNN (đã kiểm chứng)
    PI["n_v6"] = predict(STUDENT_NEW, IMGS, tag("pi_n_v6", 640), imgsz=640, device=DEVICE)
    rows = []
    for s, G in GTS.items():
        for k in PI:
            ap = [wap(*prep(PI[k], G[c], c, set(IDX), IDX), np.ones(len(IMGS))) for c in range(1, 6)]
            rows.append({"split": s, "pi": k, "Pi-only MEAN_5": np.mean(ap), **dict(zip(DEFECTS, ap))})
    pi_res = pd.DataFrame(rows); print(pi_res.round(4).to_string(index=False))
    res2, _ = system_scores({BEST_TEACHER: TEACHER[BEST_TEACHER]}, pi_name="n_v6")
    print(res2[res2.setting != "teacher only"][["split", "teacher", "MEAN_5"]].round(4).to_string(index=False))
    pi_res.to_csv(OUT / "eval_student.csv", index=False); res2.to_csv(OUT / "eval_system_student_v6.csv", index=False)

In [ ]:
# 8. UPLOAD lên Camber: <STASH_RESULTS>/<RUN_NAME>_<ngày>/ (+ dataset v6 → STASH_V6, để lần sau khỏi build)
REMOTE = f"{STASH_RESULTS}/{RUN_NAME}_{time.strftime('%Y%m%d')}" + ("_dryrun" if DRY_RUN else "")
stages = [f"teacher_{a}" for a in ARMS] + (["student_n_v6"] if TRAIN_STUDENT else [])
files = {f"{s}/{n}": RUNS / s / p for s in stages
         for n, p in [("best.pt", "weights/best.pt"), ("results.csv", "results.csv"), ("args.yaml", "args.yaml")]}
files |= {f.name: f for f in OUT.glob("eval_*.csv")} | {"progress.log": OUT / "progress.log",
                                                       "dataset_v6p5_v6_manifest.json": DS6 / "manifest.json"}
if UPLOAD_RESULTS and HAS_CAMBER:
    for dst, src_ in files.items():
        if src_.is_file(): camber_cp(str(src_), f"{REMOTE}/{dst}"); print("uploaded", dst)
    if TRAIN_STUDENT and NCNN.is_dir():
        subprocess.run([CAMBER, "stash", "cp", "-r", str(NCNN), f"{REMOTE}/student_n_v6/best_ncnn_model"], check=True,
                       stdout=subprocess.DEVNULL); print("uploaded student_n_v6/best_ncnn_model")
    print("kết quả trên Camber:", REMOTE)
    if UPLOAD_DATASET_V6:
        r = subprocess.run([sys.executable, str(REPO / "scripts/package_dataset.py"), str(DS6), "--out", str(DATA_DIR / "packages"),
                            "--upload", STASH_V6], capture_output=True, text=True)
        print("dataset v6 →", STASH_V6, "OK" if r.returncode == 0 else f"LỖI: {r.stderr[-500:]}")
else:
    print("bỏ qua upload (UPLOAD_RESULTS tắt hoặc thiếu Camber CLI / CAMBER_API_KEY) — kết quả ở", OUT)

In [ ]:
# 9. KẾT THÚC: tóm tắt + (tuỳ chọn) tắt máy để không tốn tiền GPU idle
print(f"\n===== XONG {time.strftime('%Y-%m-%d %H:%M')} — kết quả: {OUT}")
for f in ["eval_teachers_compare.csv", "eval_student.csv"]:
    if (OUT / f).is_file(): print(f"\n{f}\n" + pd.read_csv(OUT / f).round(4).to_string(index=False))
if AUTO_SHUTDOWN and not DRY_RUN and not WINDOWS:
    print("tắt máy sau 5 phút (huỷ: sudo shutdown -c)"); subprocess.run(["sudo", "shutdown", "-h", "+5"])